# Experiment 1 — Actor size on one circuit

> How does actor-network capacity affect final driving performance, interaction
> efficiency, convergence reliability and computational cost on one fixed
> circuit?

This notebook is the executable form of the Experiment 1 section of
[`docs/EXPERIMENT.md`](../docs/EXPERIMENT.md). It states the design, runs the
matrix, and presents every required output. It does not restate the learning
rules themselves: the bounded Gaussian policy, the return conventions and the
optimizer contract are in [`docs/LEARNING.md`](../docs/LEARNING.md), and the
racing MDP and circuit model are in [`docs/MDP.md`](../docs/MDP.md) and
[`docs/TRACK.md`](../docs/TRACK.md).

**What is being varied, and what is not.** One thing changes across the matrix:
the width of the two hidden layers of the actor. Everything else — circuit,
observation, reward, physics, episode limit, critic width, learning rate,
interaction budget, evaluation schedule — is held fixed, so a difference in
outcome is attributable to capacity rather than to the conditions around it.

The algorithm comparison is **secondary**. Running the same size ladder under
REINFORCE, A2C+GAE and PPO describes the practical effect of adding a critic,
GAE and bounded sample reuse, but the protocol does not assume that the more
elaborate algorithm must win, and contrary evidence is reported as it is found.


## Hypotheses

The protocol commits to four, none of which asserts a direction for capacity:

- **Capacity.** Actor size can change final task performance; larger is *not*
  assumed to be better. A wider actor has more parameters to move with the same
  number of gradient steps, which can help or hurt.
- **Efficiency.** Actor size can change the interactions and the computation
  needed to reach the task threshold. These are two different costs and are
  reported separately: a policy can be cheap in interactions and expensive in
  wall time, or the reverse.
- **Reliability.** Actor size can change the fraction of roots that learn a
  stable lap-completing policy. With five roots this is a count out of five and
  is always reported as such, never as a rate with the denominator hidden.
- **Algorithm.** A2C+GAE and PPO are expected to reduce variance or improve
  sample use relative to REINFORCE.

## Design matrix

One independent unit is a complete training run identified by
$(\text{algorithm}, \text{actor size}, \text{root identity})$.

| Algorithm | `(8, 8)` | `(32, 32)` | `(64, 64)` | `(256, 256)` |
|---|---:|---:|---:|---:|
| REINFORCE | 5 roots | 5 roots | 5 roots | 5 roots |
| A2C+GAE | 5 roots | 5 roots | 5 roots | 5 roots |
| PPO | 5 roots | 5 roots | 5 roots | 5 roots |

$3 \times 4 \times 5 = 60$ runs: 45 retained original runs plus 15 new tiny-actor runs. Root identities `0..4` are **paired across
actor sizes and algorithms**: root 2 names the same derived seed streams
wherever it appears, so a within-root difference removes the root-to-root
variation that otherwise dominates a five-sample comparison.


## Fixed conditions

Every run uses the saved `tracks/experiment_1.json` circuit and its canonical
start; the Frenet observation $(d_t, \phi_{e,t}, v_t, \delta_t, \bar\kappa_t)$; the same
action mapping, reward, episode limit and frozen physics version; one fixed
`(64, 64)` critic for A2C and PPO; the learning rate selected for its algorithm
before the experiment; 2,000,000 training interactions; deterministic
evaluation every 50,000 interactions; and checkpoints every 250,000
interactions and at the final budget.

**Selected learning rates** (from the pre-experiment configuration check, which
is a separate exercise recorded in `docs/EXPERIMENT.md`): REINFORCE $10^{-3}$;
A2C $(10^{-3}, 3\cdot 10^{-3})$; PPO $(3\cdot 10^{-4}, 10^{-2})$.

**One interaction is one call to `RacingEnv.step`**, whatever happens inside it.
Evaluation interactions are counted separately and never enter the training
budget, so the budget means the same thing for an algorithm that evaluates often
and one that does not.

## Evaluation and the convergence rule

Evaluation runs the deterministic policy $A_t = \tanh(\mu_\theta(O_t))$ from the
canonical start at zero speed, even though training samples its start pose all
around the circuit. Holding the evaluation start fixed keeps every reported
number an answer to the same question. One episode per checkpoint is enough
because the policy, the start and the circuit are all deterministic — earlier
runs took sixteen and reported a standard deviation of exactly zero.

**Stable convergence** is the first of three consecutive evaluations that
complete the lap in at most $34$ simulated seconds. That is about $1.5\times$
the reference controller's $22.3\,\mathrm{s}$ average, between its slowest lap
and the $40\,\mathrm{s}$ episode cap. It is a project choice, not a measurement.
A run that never meets it is **right-censored** at the budget and stays in every
success-rate and learning-curve summary; it is never quietly dropped.

Evaluation records the episode *outcome*, not just its return, because a return
alone cannot separate crashing from idling from lapping slowly.

**Timeout limitation.** Experiment 1 retains its original treatment: REINFORCE
stops its return at the task deadline, while A2C and PPO bootstrap once from the
critic. This does not vary with actor size, but it limits direct algorithm
comparisons.


## Configuration

`REHEARSAL` below is the only switch in this notebook. Both configurations are
written out rather than one being a commented-out alternative, so what a
rehearsal changes is visible at a glance: the budget, the evaluation interval,
the number of roots — and the run category, which is what keeps the two kinds of
output from ever being confused.

**Nothing else in this notebook depends on the choice.** The matrix, the
analysis and every table below are written once and read whatever was run.


In [ ]:
import sys
import warnings
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "experiments"))
warnings.filterwarnings(
    "ignore",
    message="pkg_resources is deprecated as an API.*",
    category=UserWarning,
    module="pygame.pkgdata",
)

from experiment_1 import PROTOCOL, REHEARSAL, analyze, contract, specifications
from matrix import execute, summarize
from reporting import read_table, show_figure, show_table

USE_REHEARSAL = False
ACTOR_NAMES = ("tiny",)
SCALE = REHEARSAL if USE_REHEARSAL else PROTOCOL
RESULTS_ROOT = SCALE.results_root
ANALYSIS_ROOT = SCALE.analysis_root
CONTRACT = contract()
SPECIFICATIONS = specifications(SCALE, ACTOR_NAMES)

print(f"budget {SCALE.budget:,} | roots {SCALE.roots} | {SCALE.category.value}")
print(f"{len(SPECIFICATIONS)} scheduled runs -> {RESULTS_ROOT}")


## Building the matrix

Each specification names a run and says how to start it; nothing is executed
here. The run identity carries the algorithm, the actor size, the observation
and the root, which is exactly the tuple that defines an independent unit, and
the directory is named after it so the results tree can be read without
consulting a manifest.


In [ ]:
show_table(
    [
        {
            "run_id": specification.run_id,
            "already complete": (specification.path / "completion.json").is_file(),
        }
        for specification in SPECIFICATIONS
    ],
    title=f"{len(SPECIFICATIONS)} scheduled runs",
)


## Running the matrix

This is the long cell. It is **resumable**: a run whose `completion.json` exists
is skipped, so an interrupted matrix continues where it stopped, and re-running
the notebook after everything is finished costs nothing.

The skip is *checked*. A finished run whose recorded reward, physics, circuit
geometry or discount no longer matches the current configuration is re-run
instead of reused, so a constant that changes mid-project cannot leave a results
tree that mixes two contracts and still looks complete.

A failing run does not stop the queue. Losing a nine-hour run of forty-five
specifications to one bad one would be worse than finishing the rest and
repairing it afterwards, which is what the protocol's failure rule asks for: an
exception is an *operational* failure, so repair the cause and rerun the same
specification. A valid run that merely learns badly is a scientific outcome and
stays.


In [ ]:
OUTCOMES = execute(SPECIFICATIONS, contract=CONTRACT)
FAILURES = summarize(OUTCOMES)
assert FAILURES == 0, f"{FAILURES} run(s) failed; see the tracebacks above."


## Analysis

Every table and figure below is regenerated from the raw run records by
`analyze_results`, and written to disk before it is displayed. Console output
and hand-copied values are not authoritative; the files are.

The analysis reads only the recorded documents — no checkpoint is loaded and no
directory name is parsed — because each evaluation record already carries its
cumulative collection and optimization time, and each circuit outcome already
carries its frozen geometry.


In [ ]:
from analyze_results import analyze_results

MANIFEST = analyze_results(
    results_root=RESULTS_ROOT,
    output_directory=ANALYSIS_ROOT,
    experiment=1,
    category=RUN_CATEGORY,
)
print(f"analyzed {len(MANIFEST['inputs'])} runs -> {ANALYSIS_ROOT}")

INVENTORY = read_table(ANALYSIS_ROOT, "run_inventory")
SUMMARIES = read_table(ANALYSIS_ROOT, "run_summaries")
CELLS = read_table(ANALYSIS_ROOT, "cell_summaries")
CURVES = read_table(ANALYSIS_ROOT, "learning_curves")
PAIRED = read_table(ANALYSIS_ROOT, "paired_summaries")
UPDATES = read_table(ANALYSIS_ROOT, "optimization_diagnostics")


### Output 1 — Configuration and parameter counts

The parameter count is the quantity the capacity hypothesis is about, so it is
reported before any performance number. The actor counts differ by design; the
critic is fixed at `(64, 64)` for both actor-critic algorithms, which isolates
actor width but may constrain the largest actor — a limitation the protocol
states rather than hides.


In [ ]:
show_table(
    INVENTORY,
    columns=[
        "run_id", "algorithm", "actor_name", "root_identity",
        "actor_parameters", "critic_parameters", "total_parameters",
        "training_interactions",
    ],
    sort_by=["algorithm", "actor_parameters", "root_identity"],
    title="Run inventory and parameter counts",
)


### Outputs 2 and 3 — Final task metrics and completion counts

Conclusions are drawn in the protocol's order: **first** how many roots finish a
lap, **then** the lap time with its completion denominator, **then** return and
progress, **then** the crash count. Return is deliberately not first — a shaped
return can rise while the car still never finishes, so the completion count is
the outcome that anchors the rest.

Every cell shows all its roots. With five roots an interval is descriptive, and
the raw points are the honest evidence.


In [ ]:
MANIFEST = analyze(SCALE)

INVENTORY = read_table(ANALYSIS_ROOT, "run_inventory")
SUMMARIES = read_table(ANALYSIS_ROOT, "run_summaries")
CELLS = read_table(ANALYSIS_ROOT, "cell_summaries")
CURVES = read_table(ANALYSIS_ROOT, "learning_curves")
PAIRED = read_table(ANALYSIS_ROOT, "paired_summaries")
UPDATES = read_table(ANALYSIS_ROOT, "optimization_diagnostics")


### Output 4 — Learning curves and normalized curve area

Curves align on **training interactions**, never on episodes or updates: an
episode means a different amount of experience in each algorithm, and an update
means a very different amount between REINFORCE and PPO.

Normalized curve area is the trapezoidal area from the first recorded evaluation
to the last, divided by that interaction span. It rewards reaching a level early
and holding it, which is the efficiency question; final performance alone cannot
distinguish a policy that arrived quickly from one that arrived at the end.
Smoothing anywhere in this notebook is display-only and never enters an area or
a convergence decision.


In [ ]:
show_figure(ANALYSIS_ROOT, "learning_curves")
show_table(
    CELLS,
    columns=[
        "algorithm", "actor_name",
        "return_auc_mean", "return_auc_sample_standard_deviation",
        "progress_auc_mean", "progress_auc_sample_standard_deviation",
    ],
    sort_by=["algorithm", "actor_name"],
    title="Normalized curve area",
)


### Output 5 — Convergence, cost and censoring

`converged` is the protocol rule: three consecutive evaluations completing the
lap within 34 simulated seconds. `censored` marks a run that never met it and
was cut off at the budget. Convergence summaries always show the success
fraction and the censoring rather than averaging only the runs that succeeded,
because averaging survivors would report the best case as if it were the
typical one.

Both costs of reaching the threshold are shown: interactions, which is sample
efficiency, and wall seconds, which is computation. They can disagree.


In [ ]:
show_table(
    SUMMARIES,
    columns=[
        "algorithm", "actor_name", "root_identity", "converged", "censored",
        "convergence_interactions", "convergence_duration", "episodes_to_convergence",
    ],
    sort_by=["algorithm", "actor_name", "root_identity"],
    title="Convergence with censoring",
)
show_figure(ANALYSIS_ROOT, "convergence_resources")


### Output 6 — Performance and cost against actor parameter count

This is the capacity hypothesis plotted directly. If wider were simply better
the points would rise monotonically with parameters; the protocol does not
assume they do.


In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)
for algorithm in ("reinforce", "a2c", "ppo"):
    rows = sorted(
        (row for row in SUMMARIES if row["algorithm"] == algorithm),
        key=lambda row: row["actor_parameters"],
    )
    parameters = [row["actor_parameters"] for row in rows]
    axes[0].scatter(parameters, [row["final_mean_return"] for row in rows], label=algorithm)
    axes[1].scatter(parameters, [row["final_mean_progress"] for row in rows], label=algorithm)
    axes[2].scatter(parameters, [row["end_to_end_duration"] / 60 for row in rows], label=algorithm)
for axis, title, ylabel in zip(
    axes,
    ("Final return", "Final progress", "End-to-end cost"),
    ("deterministic return", "normalized progress", "minutes"),
):
    axis.set(title=title, xlabel="actor parameters", ylabel=ylabel, xscale="log")
    axis.grid(alpha=0.3)
    axis.legend()
plt.show()


### Output 7 — Throughput, memory and end-to-end runtime

Collection throughput and optimization time separate the two things that make a
run slow. A wider actor costs more per gradient step but not per environment
step, so the two columns move differently with capacity.


In [ ]:
show_table(
    SUMMARIES,
    columns=[
        "algorithm", "actor_name", "root_identity",
        "collection_throughput", "collection_duration", "optimization_duration",
        "evaluation_duration", "end_to_end_duration", "peak_process_memory",
    ],
    sort_by=["algorithm", "actor_name", "root_identity"],
    title="Computational cost",
)


### Output 8 — Optimization diagnostics

These are the internals that explain *why* a curve looks as it does: gradient
and update norms, the learned exploration scale, and for the actor-critic pair
the critic's explained variance, the advantage scale, and for PPO the
importance-ratio distribution, approximate KL and clip fraction.


In [ ]:
show_figure(ANALYSIS_ROOT, "optimization_diagnostics")
show_table(
    UPDATES,
    columns=[
        "run_id", "training_interactions", "actor_loss", "actor_gradient_norm",
        "entropy_proxy", "log_standard_deviation_0", "explained_variance",
    ],
    sort_by=["run_id", "training_interactions"],
    limit=15,
    title="Optimization diagnostics (first rows; the full table is on disk)",
)


### Output 9 — Curvature-conditioned driving

Return says how well a policy scored; this says what it actually *did*. Speed,
throttle and steering are conditioned on the curvature the car was in at the
time, which separates a policy that brakes for corners from one that drives
slowly everywhere.

The representative policy per cell is chosen by a rule fixed in advance — the
root closest to its cell's median final return, with the lower root identity
breaking an exact tie — so the illustration is not selected to flatter.


In [ ]:
show_figure(ANALYSIS_ROOT, "curvature_controls")
show_figure(ANALYSIS_ROOT, "task_outcomes")


## PPO actor selection for Experiment 2

Experiment 2 needs one PPO actor width, and the protocol fixes how it is chosen
so the choice cannot be made by looking at Experiment 2. **Only Experiment 1 PPO
results are used**, and the calculation is recorded before any test circuit is
opened.

The rule has four steps:

1. find the size with the highest mean final deterministic return;
2. compute paired root-level return deficits for every other size;
3. admit a size whose mean deficit is no greater than one standard error **and**
   whose completion count is no more than one root below the best size;
4. choose the admitted actor with the fewest parameters.

Step 3 is why this is a rule and not `argmax`. It admits on *equivalence within
noise*, so a smaller actor that merely fails to be strictly best is not
discarded. Note what that implies: a deficit that is small but perfectly
consistent across roots has a standard error of zero and is therefore **not**
admitted, which is the correct reading — a difference with no variance is
certain, however small it is.

The whole calculation is written out, not just its answer, because the
calculation itself is a required Experiment 2 output.


In [ ]:
import json

selection_path = ANALYSIS_ROOT / "ppo_actor_selection.json"
selection = json.loads(selection_path.read_text(encoding="utf-8"))
SELECTION = selection["candidates"]
SELECTED_ACTOR = selection["selected_actor"]

show_table(
    SELECTION,
    columns=[
        "actor_name",
        "actor_parameters",
        "paired_root_count",
        "mean_final_return",
        "completion_count",
        "is_best_mean_return",
        "mean_paired_deficit",
        "paired_deficit_standard_error",
        "within_one_standard_error",
        "completion_within_one_root",
        "admitted",
        "selected",
    ],
    title="Original PPO actor-size selection",
)
print(f"Experiment 2 uses the original selected actor: {SELECTED_ACTOR!r}")
print(f"Recorded at {selection_path}")


## Limitations

Stated by the protocol, and unchanged by any result above:

- Five roots give only a modest estimate of training variation, so intervals
  here are descriptive and conclusions rest on magnitudes and raw outcomes.
- One circuit makes every conclusion circuit-specific. Experiment 2 is what
  addresses generalization.
- Changing width at fixed two-layer depth does not cover every notion of
  network complexity.
- The fixed `(64, 64)` critic isolates actor width but may constrain the
  largest actor.
- Algorithm differences cannot be attributed to abstract complexity alone,
  because the three differ in estimator *and* update schedule at once.
